# TASK 4: Corporate Financial Performance & Scenario Intelligence
### Financial Diagnostics, COGS Analysis, Margin Rankings, and What-If Parameterized Scenario Simulator

**Author**: Vijay Maheswari  
**Framework**: RISK Methodology — Simulation & KPI Verification  
**Dataset**: Corporate Financial Sample (700 transaction batches, $118.7M Sales)


In [1]:
import sys
import os

for p in [os.path.abspath(".."), os.path.abspath(".")]:
    if p not in sys.path:
        sys.path.insert(0, p)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.loaders import load_financial_sample
from src.cleaning import clean_financial_sample
from src.metrics import calculate_financial_kpis, simulate_financial_scenario
from src.validation import validate_financial_schema

sns.set_theme(style="whitegrid", palette="muted")


## 1. Load and Standardize Financial Sample Data
We ingest the raw Excel workbook, strip whitespaces from column headers (e.g. `' Sales'` -> `'Sales'`), and audit required fields.


In [2]:
raw_df = load_financial_sample()
print(f"Raw financial shape: {raw_df.shape}")

# Validate and clean
df = clean_financial_sample(raw_df)
audit = validate_financial_schema(df)
print(audit.summary())

# Export processed FinancialFact table
df.to_csv("data/processed/FinancialFact.csv", index=False)
print("Saved data/processed/FinancialFact.csv")


Raw financial shape: (700, 16)
[PASSED] Corporate Financial Sample: 700 rows, 22 cols
Saved data/processed/FinancialFact.csv


## 2. Core Financial Scorecard
We calculate Grand Total Sales, COGS, Profit, and standard accounting ratios.


In [3]:
kpis = calculate_financial_kpis(df)
print(f"Total Gross Sales   : ${kpis['Total Gross Sales']:,.2f}")
print(f"Total Discounts     : ${kpis['Total Discounts']:,.2f} ({kpis['Effective Discount %']:.2f}% Effective Rate)")
print(f"Total Net Sales     : ${kpis['Total Sales']:,.2f}")
print(f"Total COGS          : ${kpis['Total COGS']:,.2f} ({kpis['COGS Ratio %']:.2f}% of Net Sales)")
print(f"Total Operating Profit: ${kpis['Total Profit']:,.2f}")
print(f"Gross Profit Margin : {kpis['Gross Profit Margin %']:.2f}%")


Total Gross Sales   : $127,931,598.50
Total Discounts     : $9,205,248.24 (7.20% Effective Rate)
Total Net Sales     : $118,726,350.26
Total COGS          : $101,832,648.00 (85.77% of Net Sales)
Total Operating Profit: $16,893,702.26
Gross Profit Margin : 14.23%


## 3. Segment & Country Commercial Breakdown
Diagnostics reveal that Enterprise operates at a net loss, while Channel Partners and Government drive substantial margins.


In [4]:
print("Customer Segment Breakdown:")
print(kpis["By Segment"].to_string(index=False))
print("Geographic Breakdown:")
print(kpis["By Country"].to_string(index=False))


Customer Segment Breakdown:
         Segment        Sales       COGS       Profit    Units  Profit Margin %
Channel Partners  1800593.640   483790.5  1316803.140 161263.5        73.131611
      Enterprise 19611694.375 20226240.0  -614545.625 168552.0        -3.133567
      Government 52504260.670 41116087.5 11388173.170 470673.5        21.689998
       Midmarket  2381883.075  1721780.0   660103.075 172178.0        27.713496
  Small Business 42427918.500 38284750.0  4143168.500 153139.0         9.765194
Geographic Breakdown:
                 Country        Sales       COGS      Profit  Profit Margin %
                  Canada 24887654.885 21358426.0 3529228.885        14.180641
                  France 24354172.280 20573151.5 3781020.780        15.525146
                 Germany 23505340.820 19824952.0 3680388.820        15.657671
                  Mexico 20949352.110 18041829.0 2907523.110        13.878821
United States of America 25029830.165 22034289.5 2995540.665        11.967883


## 4. Visual Diagnostics Generation
Render and export all 5 financial diagnostic and scenario charts.


In [5]:
from src.generate_financial_intelligence import generate_financial_visualizations

generate_financial_visualizations(df, "outputs/figures")
print("Visualizations saved to outputs/figures/")


D:\intership\Datheon_Data_Analytics_&_Power_BI_Internship\vj\src\generate_financial_intelligence.py:62: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=c_agg, x="Country", y="Profit", palette="Greens_d")


D:\intership\Datheon_Data_Analytics_&_Power_BI_Internship\vj\src\generate_financial_intelligence.py:77: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=prod_agg, x="Product", y="COGS Ratio %", palette="Reds_d")


Visualizations saved to outputs/figures/


## 5. Financial What-If Scenario Simulator
We simulate multi-parameter shocks:
- Scenario A: 10% Discount Reduction, 3% COGS inflation, 5% Volume expansion.
- Scenario B: Severe Supply-Chain Stress (10% COGS inflation, 0% volume change).


In [6]:
# Scenario A: Balanced Operational Optimization
sim_a = simulate_financial_scenario(df, discount_pct_delta=-0.10, cogs_pct_delta=0.03, volume_pct_delta=0.05)
print("Scenario A Results:")
print("Baseline Profit  : $" + f"{sim_a['baseline']['Total Profit']:,.2f}")
print("Simulated Profit : $" + f"{sim_a['simulated']['Total Profit']:,.2f}")
print("Profit Variance  : $" + f"{sim_a['variance']['Profit Delta ($)']:,.2f} ({sim_a['variance']['Profit Delta (%)']:+.2f}%)")

# Scenario B: Supply Chain Inflation Shock
sim_b = simulate_financial_scenario(df, discount_pct_delta=0.0, cogs_pct_delta=0.10, volume_pct_delta=0.0)
print("Scenario B Results (10% COGS Inflation):")
print("Baseline Profit  : $" + f"{sim_b['baseline']['Total Profit']:,.2f}")
print("Simulated Profit : $" + f"{sim_b['simulated']['Total Profit']:,.2f}")
print("Profit Variance  : $" + f"{sim_b['variance']['Profit Delta ($)']:,.2f} ({sim_b['variance']['Profit Delta (%)']:+.2f}%)")


Scenario A Results:
Baseline Profit  : $16,893,702.26
Simulated Profit : $15,497,210.03
Profit Variance  : $-1,396,492.23 (-8.27%)
Scenario B Results (10% COGS Inflation):
Baseline Profit  : $16,893,702.26
Simulated Profit : $6,710,437.46
Profit Variance  : $-10,183,264.80 (-60.28%)
